# Border 2 Fire — Real-Data EnKF Experiment

Uses **real WFIGS perimeters** and **derived UTC observation timestamps** anchored
to CAL FIRE incident reports and NASA FIRMS hotspot detections.

| Obs | UTC timestamp | Acres | Source |
|-----|--------------|-------|--------|
| t0 | 2025-01-23 18:22 | ~first detection | FIRMS Landsat 9 |
| t1 | 2025-01-24 20:36 | 4 250 | CAL FIRE 12:36 PM PT update |
| t2 | 2025-01-25 03:49 | 6 273 | CAL FIRE 7:49 PM PT update |
| t3 | 2025-01-27 00:12 | 6 625 | CAL FIRE 4:12 PM PT update |

Calls `adjusted_state_EnKF_farsite` which now uses `forward_pass_farsite_multistep` internally.
Logs three CSV files for paper analysis: `members.csv`, `steps.csv`, `prep.csv`.

**Setup**: There is no need to re-run these cells if install_packages.sh has already been executed once.

In [1]:
!chmod +x src/TestFARSITE src/lcpmake install_packages.sh

In [2]:
!bash install_packages.sh

Using: /opt/conda/envs/gdal_env/bin/python
/opt/conda/envs/gdal_env/lib/python3.11/site-packages/IPython/paths.py:60: UserWarning: IPython dir '/home/jovyan/.ipython' is not a writable location, using a temp directory.
  warn("IPython dir '{}' is not a writable location,"
Installed kernelspec gdal_env in /home/jovyan/.local/share/jupyter/kernels/gdal_env
Done.


## Imports and Configuration
**Important**: Switch to the Python kernel named Python (gdal_env) before proceeding with the rest of the notebook.

In [3]:
import sys
sys.path.insert(0, 'src')

import csv
import time
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from enkf import adjusted_state_EnKF_farsite
from geometry import geom_to_state, state_to_geom, align_states
from config import OUTPUT_DIR, DATA_DIR, DEFAULT_DIST_RES, DEFAULT_PERIM_RES
from firemap import query_weather_for_timestep, fetch_fire_perimeters

print('Imports OK')

Imports OK


## Observation Timestamps

These UTC timestamps were derived by cross-referencing CAL FIRE incident update
acreage reports with the WFIGS perimeter acreages, and anchoring `t0` to the
first NASA FIRMS Landsat 9 hotspot detection.

In [4]:
OBS_TIMESTAMPS = [
    pd.Timestamp('2025-01-23 18:22:00', tz='UTC'),  # t0 — FIRMS ignition anchor
    pd.Timestamp('2025-01-24 20:36:00', tz='UTC'),  # t1 — CAL FIRE 12:36 PM PT (4,250 ac)
    pd.Timestamp('2025-01-25 03:49:00', tz='UTC'),  # t2 — CAL FIRE 7:49 PM PT  (6,273 ac)
    pd.Timestamp('2025-01-27 00:12:00', tz='UTC'),  # t3 — CAL FIRE 4:12 PM PT  (6,625 ac)
]

print('Observation timestamps:')
for i, ts in enumerate(OBS_TIMESTAMPS):
    print(f'  t{i}: {ts}')

Observation timestamps:
  t0: 2025-01-23 18:22:00+00:00
  t1: 2025-01-24 20:36:00+00:00
  t2: 2025-01-25 03:49:00+00:00
  t3: 2025-01-27 00:12:00+00:00


## Load Real Perimeters

In [5]:
PERIMETERS_PATH = DATA_DIR / 'border2_inputs' / 'border_2_perimeters.geojson'
LCP_PATH = str(DATA_DIR / 'border2_inputs' / 'landscape.lcp')
FIRE_NAME = 'Border_2_Real'

raw_gdf = gpd.read_file(PERIMETERS_PATH)
print(f'Raw perimeters: {len(raw_gdf)}')
print(raw_gdf[['datetime', 'acres']].to_string())

Raw perimeters: 6
    datetime        acres
0 2025-01-23          NaN
1 2025-01-23   567.000000
2 2025-01-23   566.339868
3 2025-01-24  4234.873545
4 2025-01-24  6272.660000
5 2025-01-26  6624.770000


ERROR 1: PROJ: proj_create_from_database: Open of /opt/conda/envs/gdal_env/share/proj failed


### Assign Precise Timestamps and Reproject to EPSG:5070

WFIGS perimeters are in WGS84. FARSITE expects EPSG:5070 (Albers Equal Area).
Reprojecting here ensures `geom_to_state` stores EPSG:5070 coordinates.

In [6]:
# Select 4 representative perimeters (one per observation time)
SELECTED_IDX = [1, 3, 4, 5]

perimeters_gdf = raw_gdf.iloc[SELECTED_IDX].copy().reset_index(drop=True)
perimeters_gdf['obs_time'] = OBS_TIMESTAMPS
perimeters_gdf['obs_time'] = pd.to_datetime(perimeters_gdf['obs_time'], utc=True)

# Reproject to EPSG:5070 so geometry coordinates match LCP extent
perimeters_gdf = perimeters_gdf.to_crs('EPSG:5070')
print('CRS after reprojection:', perimeters_gdf.crs)

print('Selected perimeters with precise timestamps:')
for i, row in perimeters_gdf.iterrows():
    print(f'  [{i}] {row["obs_time"]}  bounds={row.geometry.bounds}')

CRS after reprojection: EPSG:5070
Selected perimeters with precise timestamps:
  [0] 2025-01-23 18:22:00+00:00  bounds=(-1932307.9862038032, 1270177.2506641382, -1930058.200045176, 1271825.3811061257)
  [1] 2025-01-24 20:36:00+00:00  bounds=(-1935511.6878793458, 1270171.4925100275, -1929892.4050902296, 1276188.986916912)
  [2] 2025-01-25 03:49:00+00:00  bounds=(-1935891.5206762566, 1270171.4808392834, -1929222.091653914, 1277003.6968184116)
  [3] 2025-01-27 00:12:00+00:00  bounds=(-1935891.5206762566, 1270171.4808392834, -1929222.091653914, 1277003.6968184116)


## Experiment Parameters

In [7]:
DIST_RES    = DEFAULT_DIST_RES   # meters (default 150)
PERIM_RES   = DEFAULT_PERIM_RES  # meters (default 150)

N_SAMPLES   = 300
N_VERTEX    = 20
VSIZE       = 300.0  # observation noise std (m)
WSIZE       = 100.0  # model noise std (m)
RANDOM_SEED = 1234

# Default temperature/humidity for FARSITE (no API source)
DEFAULT_TEMPERATURE = 66    # °F
DEFAULT_HUMIDITY    = 0.08  # fraction

# Weather location (centroid of fire area in WGS84)
WX_LAT = 32.59588662537945
WX_LON = -116.85708741551528

rng = np.random.Generator(np.random.PCG64(RANDOM_SEED))

print('Parameters:')
print(f'  DIST_RES={DIST_RES}m  PERIM_RES={PERIM_RES}m')
print(f'  N_SAMPLES={N_SAMPLES}  N_VERTEX={N_VERTEX}')
print(f'  VSIZE={VSIZE}m  WSIZE={WSIZE}m')
print(f'  LCP_PATH={LCP_PATH}')
print(f'  LCP exists: {Path(LCP_PATH).exists()}')

Parameters:
  DIST_RES=150m  PERIM_RES=150m
  N_SAMPLES=300  N_VERTEX=20
  VSIZE=300.0m  WSIZE=100.0m
  LCP_PATH=/home/jovyan/work/digital-twin-group/wildfire-digital-twin/data/border2_inputs/landscape.lcp
  LCP exists: True


## CSV Loggers

In [8]:
LOG_DIR = OUTPUT_DIR / FIRE_NAME
LOG_DIR.mkdir(parents=True, exist_ok=True)

MEMBERS_CSV = LOG_DIR / 'members.csv'
STEPS_CSV   = LOG_DIR / 'steps.csv'
PREP_CSV    = LOG_DIR / 'prep.csv'

MEMBERS_FIELDS = ['run_id','step','member','ws','wd','farsite_ok','elapsed_s','forecast_area_ac']
STEPS_FIELDS   = ['step','t_start','t_end','dt_hours','valid_members','total_members',
                   'obs_area_ac','analysis_area_ac','elapsed_s','mean_ws','mean_wd']
PREP_FIELDS    = ['action','step','detail','elapsed_s','status']

members_f = open(MEMBERS_CSV, 'w', newline='')
steps_f   = open(STEPS_CSV,   'w', newline='')
prep_f    = open(PREP_CSV,    'w', newline='')

members_w = csv.DictWriter(members_f, fieldnames=MEMBERS_FIELDS)
steps_w   = csv.DictWriter(steps_f,   fieldnames=STEPS_FIELDS)
prep_w    = csv.DictWriter(prep_f,    fieldnames=PREP_FIELDS)

members_w.writeheader()
steps_w.writeheader()
prep_w.writeheader()

print(f'CSV loggers initialised in {LOG_DIR}')

CSV loggers initialised in /home/jovyan/work/digital-twin-group/wildfire-digital-twin/outputs/Border_2_Real


## Initialise EnKF State

In [9]:
_t0 = time.time()
current_state = geom_to_state(perimeters_gdf['geometry'].iloc[0])
X = 1000.0 * np.eye(2 * N_VERTEX)

prep_w.writerow(dict(
    action='init_state', step=-1,
    detail=f'initial perimeter -> {N_VERTEX} vertices, cov=1000*I',
    elapsed_s=round(time.time()-_t0, 3), status='ok'
))

print(f'State dimension: {2*N_VERTEX}')
print(f'Covariance shape: {X.shape}')

State dimension: 40
Covariance shape: (40, 40)


## Results Storage

In [10]:
results = {
    'step':            [],
    't_start':         [],
    't_end':           [],
    'initial_geom':    [],
    'observed_geom':   [],
    'analysis_geom':   [],
    'covariance':      [],
    'valid_members':   [],
    'mean_ws':         [],
    'mean_wd':         [],
    'zkphat_ensemble': [],
    'xkphat_ensemble': [],
}

## Sequential Assimilation Loop

For each pair of consecutive observations:
1. Query weather for the interval
2. Sample wind speed/direction for the ensemble
3. `adjusted_state_EnKF_farsite` — FARSITE forward model + EnKF update
4. Log to `members.csv`, `steps.csv`

In [ ]:
n_timesteps = len(perimeters_gdf) - 1
print(f'Starting sequential assimilation for {n_timesteps} timesteps...')
print('=' * 70)

for t in range(n_timesteps):
    step_start = time.time()
    print(f'\n{"="*70}')
    print(f'STEP {t}  ->  {t+1}')
    print(f'{"="*70}')

    current_obs = perimeters_gdf.iloc[t]
    next_obs    = perimeters_gdf.iloc[t + 1]
    t_start     = current_obs['obs_time']
    t_end       = next_obs['obs_time']
    dt          = t_end - t_start

    print(f'  t_start : {t_start}')
    print(f'  t_end   : {t_end}')
    print(f'  dt      : {dt}')

    # -- Weather query -------------------------------------------------------
    _tw = time.time()
    try:
        ws_list, wd_list = query_weather_for_timestep(
            lat=WX_LAT, lon=WX_LON,
            start_time=t_start, end_time=t_end,
            verbose=False
        )
        mean_ws = float(np.mean(ws_list))
        mean_wd = float(np.mean(wd_list))
        wx_status = 'ok'
    except Exception as e:
        print(f'  [WARN] Weather query failed: {e}  — using fallback')
        ws_list = [10.0]
        wd_list = [225.0]
        mean_ws, mean_wd = 10.0, 225.0
        wx_status = f'fallback:{e}'
    prep_w.writerow(dict(
        action='query_weather', step=t,
        detail=f'ws={mean_ws:.1f} wd={mean_wd:.1f} n_obs={len(ws_list)}',
        elapsed_s=round(time.time()-_tw, 3), status=wx_status
    ))

    # -- Sample weather for ensemble -----------------------------------------
    if len(ws_list) > 1:
        idx = rng.integers(0, len(ws_list), size=N_SAMPLES)
        sampled_ws = np.asarray(ws_list)[idx]
        sampled_wd = np.asarray(wd_list)[idx]
    else:
        sampled_ws = mean_ws + rng.normal(0, 2.0, size=N_SAMPLES)
        sampled_wd = mean_wd + rng.normal(0, 15.0, size=N_SAMPLES)
        sampled_ws = np.maximum(sampled_ws, 0)
        sampled_wd = np.fmod(sampled_wd + 360, 360)

    # -- Align observation state ---------------------------------------------
    observation_state = geom_to_state(next_obs['geometry'])
    observation_state, current_state = align_states(
        [observation_state, current_state], vertex_count=N_VERTEX
    )
    n_states = 2 * N_VERTEX

    # -- Build params dict (includes temperature + humidity for FARSITE) -----
    # dt is passed as a pandas Timedelta; adjusted_state_EnKF_farsite
    # passes it through to forward_pass_farsite_multistep which needs it.
    enkf_params = {
        'windspeed':     int(mean_ws),
        'winddirection': int(mean_wd),
        'temperature':   DEFAULT_TEMPERATURE,
        'humidity':      DEFAULT_HUMIDITY,
        'dt':            dt,
    }

    # -- adjusted_state_EnKF_farsite -----------------------------------------
    print(f'\nRunning EnKF (N={N_SAMPLES})...')
    _te = time.time()
    (
        adjusted_state, X,
        zkphat_ensemble, xkhat_ensemble, ykhat_ensemble, xkphat_ensemble
    ) = adjusted_state_EnKF_farsite(
        initial_state     = current_state,
        observation_state = observation_state,
        observation_time  = t_end,
        X                 = X,
        n_states          = n_states,
        n_output          = n_states,
        n_vertex          = N_VERTEX,
        n_samples         = N_SAMPLES,
        rng               = rng,
        sampled_wslst     = sampled_ws,
        sampled_wdlst     = sampled_wd,
        dt                = dt,
        vsize             = VSIZE,
        wsize             = WSIZE,
        dist_res          = DIST_RES,
        perim_res         = PERIM_RES,
        lcppath           = LCP_PATH,
        params            = enkf_params
    )
    ensemble_elapsed = round(time.time() - _te, 3)

    # Count valid members (non-zero columns in zkphat_ensemble)
    valid_mask = np.any(zkphat_ensemble != 0, axis=0)
    valid_members = int(valid_mask.sum())
    zero_samples  = list(np.where(~valid_mask)[0])
    print(f'  Valid members: {valid_members}/{N_SAMPLES}  ({ensemble_elapsed:.1f}s)')

    # Log each member
    for s in range(N_SAMPLES):
        ok = int(s not in zero_samples)
        area_ac = 0.0
        if ok:
            try:
                g = state_to_geom(zkphat_ensemble[:, s:s+1])
                area_ac = round(g.area / 4046.86, 2)
            except Exception:
                area_ac = 0.0
        members_w.writerow(dict(
            run_id=f't{t}_s{s}',
            step=t, member=s,
            ws=round(float(sampled_ws[s]), 2),
            wd=round(float(sampled_wd[s]), 2),
            farsite_ok=ok,
            elapsed_s=round(ensemble_elapsed / N_SAMPLES, 4),
            forecast_area_ac=area_ac,
        ))

    analysis_geom = state_to_geom(adjusted_state)
    obs_area_ac   = round(next_obs.geometry.area / 4046.86, 1)
    anal_area_ac  = round(analysis_geom.area / 4046.86, 1)
    step_elapsed  = round(time.time() - step_start, 3)

    print(f'  Analysis area : {anal_area_ac} ac  (observed: {obs_area_ac} ac)')
    print(f'  Step elapsed  : {step_elapsed:.1f}s')

    steps_w.writerow(dict(
        step=t,
        t_start=str(t_start), t_end=str(t_end),
        dt_hours=round(dt.total_seconds()/3600, 3),
        valid_members=valid_members, total_members=N_SAMPLES,
        obs_area_ac=obs_area_ac, analysis_area_ac=anal_area_ac,
        elapsed_s=step_elapsed,
        mean_ws=round(mean_ws, 2), mean_wd=round(mean_wd, 2),
    ))

    results['step'].append(t)
    results['t_start'].append(t_start)
    results['t_end'].append(t_end)
    results['initial_geom'].append(state_to_geom(current_state))
    results['observed_geom'].append(next_obs.geometry)
    results['analysis_geom'].append(analysis_geom)
    results['covariance'].append(X.copy())
    results['valid_members'].append(valid_members)
    results['mean_ws'].append(mean_ws)
    results['mean_wd'].append(mean_wd)
    results['zkphat_ensemble'].append(zkphat_ensemble.copy())
    results['xkphat_ensemble'].append(xkphat_ensemble.copy())

    pd.to_pickle(results, LOG_DIR / f'{FIRE_NAME}_enkf_results.pkl')

    current_state = adjusted_state

print(f'\n{"="*70}')
print(f'SIMULATION COMPLETE — {len(results["step"])} steps')
print(f'{"="*70}')

Starting sequential assimilation for 3 timesteps...

STEP 0  ->  1
  t_start : 2025-01-23 18:22:00+00:00
  t_end   : 2025-01-24 20:36:00+00:00
  dt      : 1 days 02:14:00

Running EnKF (N=300)...
Initial bounds: (-1932273.2533752846, 1270261.5950693695, -1930118.2197492204, 1271794.471453636)
Observation bounds: (-1935328.1113533268, 1270363.8775455414, -1930236.3472832688, 1275866.056184982)


Running ensemble:   0%|          | 0/300 [00:00<?, ?it/s]

  Step 1/52: 2.533 km2
  Step 2/52: 2.770 km2
  Step 3/52: 3.001 km2
  Step 4/52: 3.172 km2
  Step 5/52: 3.357 km2
  Step 6/52: 3.577 km2
  Step 7/52: 3.923 km2
  Step 8/52: 4.325 km2
  Step 9/52: 4.721 km2
  Step 10/52: 5.285 km2
  Step 11/52: 5.803 km2
  Step 12/52: 6.368 km2
  Step 13/52: 6.917 km2
  Step 14/52: 7.437 km2
  Step 15/52: 8.172 km2
  Step 16/52: 9.182 km2
  Step 17/52: 10.087 km2
  Step 18/52: 10.987 km2
  Step 19/52: 11.937 km2
  Step 20/52: 12.837 km2
  Step 21/52: 13.805 km2
  Step 22/52: 14.893 km2
  Step 23/52: 16.050 km2
  Step 24/52: 17.162 km2
  Step 25/52: 18.467 km2
  Step 26/52: 20.027 km2
  Step 27/52: 21.320 km2
  Step 28/52: 22.719 km2
  Step 29/52: 24.260 km2
  Step 30/52: 25.806 km2
  Step 31/52: 27.130 km2
  Step 32/52: 28.598 km2
  Step 33/52: 30.020 km2
  Step 34/52: 31.439 km2
  Step 35/52: 32.770 km2
  Step 36/52: 34.117 km2
  Step 37/52: 35.654 km2
  Step 38/52: 37.092 km2
  Step 39/52: 38.550 km2
  Step 40/52: 40.116 km2
  Step 41/52: 41.813 km2


Running ensemble:   0%|          | 1/300 [01:26<7:09:10, 86.12s/it]

  Step 1/52: 2.558 km2
  Step 2/52: 2.786 km2
  Step 3/52: 3.002 km2
  Step 4/52: 3.201 km2
  Step 5/52: 3.391 km2
  Step 6/52: 3.662 km2
  Step 7/52: 4.064 km2
  Step 8/52: 4.615 km2
  Step 9/52: 5.091 km2
  Step 10/52: 5.557 km2
  Step 11/52: 6.108 km2
  Step 12/52: 6.623 km2
  Step 13/52: 7.163 km2
  Step 14/52: 7.994 km2
  Step 15/52: 8.811 km2
  Step 16/52: 9.706 km2
  Step 17/52: 10.520 km2
  Step 18/52: 11.448 km2
  Step 19/52: 12.513 km2
  Step 20/52: 13.454 km2
  Step 21/52: 14.611 km2
  Step 22/52: 15.621 km2
  Step 23/52: 16.681 km2
  Step 24/52: 17.770 km2
  Step 25/52: 19.045 km2
  Step 26/52: 20.364 km2
  Step 27/52: 21.624 km2
  Step 28/52: 22.843 km2
  Step 29/52: 24.164 km2
  Step 30/52: 25.652 km2
  Step 31/52: 27.307 km2
  Step 32/52: 29.062 km2
  Step 33/52: 30.434 km2
  Step 34/52: 31.822 km2
  Step 35/52: 33.149 km2
  Step 36/52: 34.557 km2
  Step 37/52: 36.015 km2
  Step 38/52: 37.454 km2
  Step 39/52: 38.958 km2
  Step 40/52: 40.863 km2
  Step 41/52: 42.588 km2


Running ensemble:   1%|          | 2/300 [02:53<7:10:50, 86.75s/it]

  Step 1/52: 2.609 km2
  Step 2/52: 2.820 km2
  Step 3/52: 3.025 km2
  Step 4/52: 3.241 km2
  Step 5/52: 3.462 km2
  Step 6/52: 3.787 km2
  Step 7/52: 4.235 km2
  Step 8/52: 4.728 km2
  Step 9/52: 5.188 km2
  Step 10/52: 5.666 km2
  Step 11/52: 6.179 km2
  Step 12/52: 6.765 km2
  Step 13/52: 7.515 km2
  Step 14/52: 8.449 km2
  Step 15/52: 9.504 km2
  Step 16/52: 10.431 km2
  Step 17/52: 11.279 km2
  Step 18/52: 12.212 km2
  Step 19/52: 13.154 km2
  Step 20/52: 14.094 km2
  Step 21/52: 15.012 km2
  Step 22/52: 16.035 km2
  Step 23/52: 17.164 km2
  Step 24/52: 18.304 km2
  Step 25/52: 19.463 km2
  Step 26/52: 20.858 km2
  Step 27/52: 21.938 km2
  Step 28/52: 23.427 km2
  Step 29/52: 25.044 km2
  Step 30/52: 26.866 km2
  Step 31/52: 28.288 km2
  Step 32/52: 29.638 km2
  Step 33/52: 30.942 km2
  Step 34/52: 32.341 km2
  Step 35/52: 33.924 km2
  Step 36/52: 35.326 km2
  Step 37/52: 36.714 km2
  Step 38/52: 38.252 km2
  Step 39/52: 39.859 km2
  Step 40/52: 41.716 km2
  Step 41/52: 43.526 km2

Running ensemble:   1%|          | 3/300 [04:28<7:28:23, 90.58s/it]

  Step 1/52: 2.591 km2
  Step 2/52: 2.838 km2
  Step 3/52: 3.039 km2
  Step 4/52: 3.246 km2
  Step 5/52: 3.579 km2
  Step 6/52: 4.070 km2
  Step 7/52: 4.721 km2
  Step 8/52: 5.509 km2
  Step 9/52: 6.202 km2
  Step 10/52: 7.095 km2
  Step 11/52: 8.625 km2
  Step 12/52: 9.887 km2
  Step 13/52: 10.878 km2
  Step 14/52: 12.176 km2
  Step 15/52: 14.010 km2
  Step 16/52: 15.307 km2
  Step 17/52: 16.677 km2
  Step 18/52: 18.029 km2
  Step 19/52: 19.272 km2
  Step 20/52: 20.642 km2
  Step 21/52: 22.111 km2
  Step 22/52: 23.775 km2
  Step 23/52: 25.427 km2
  Step 24/52: 27.125 km2
  Step 25/52: 29.039 km2
  Step 26/52: 30.991 km2
  Step 27/52: 32.922 km2
  Step 28/52: 34.906 km2
  Step 29/52: 36.794 km2
  Step 30/52: 38.965 km2
  Step 31/52: 41.305 km2
  Step 32/52: 43.546 km2
  Step 33/52: 45.297 km2
  Step 34/52: 47.061 km2
  Step 35/52: 48.818 km2
  Step 36/52: 50.552 km2
  Step 37/52: 52.286 km2
  Step 38/52: 53.951 km2
  Step 39/52: 55.390 km2
  Step 40/52: 56.711 km2
  Step 41/52: 57.888 

Running ensemble:   1%|▏         | 4/300 [06:04<7:36:46, 92.59s/it]

  Step 1/52: 2.590 km2
  Step 2/52: 2.824 km2
  Step 3/52: 3.031 km2
  Step 4/52: 3.273 km2
  Step 5/52: 3.661 km2
  Step 6/52: 4.184 km2
  Step 7/52: 4.810 km2
  Step 8/52: 5.428 km2
  Step 9/52: 6.212 km2
  Step 10/52: 6.982 km2
  Step 11/52: 7.999 km2
  Step 12/52: 9.164 km2
  Step 13/52: 10.352 km2
  Step 14/52: 11.447 km2
  Step 15/52: 12.780 km2
  Step 16/52: 14.544 km2
  Step 17/52: 15.997 km2
  Step 18/52: 17.285 km2
  Step 19/52: 18.486 km2
  Step 20/52: 19.793 km2
  Step 21/52: 21.112 km2
  Step 22/52: 22.595 km2
  Step 23/52: 24.250 km2
  Step 24/52: 25.827 km2
  Step 25/52: 27.622 km2
  Step 26/52: 29.439 km2
  Step 27/52: 31.239 km2
  Step 28/52: 33.033 km2
  Step 29/52: 34.884 km2
  Step 30/52: 36.936 km2
  Step 31/52: 39.278 km2
  Step 32/52: 41.405 km2
  Step 33/52: 43.525 km2
  Step 34/52: 45.747 km2
  Step 35/52: 48.219 km2


## Finalise Logs

In [ ]:
members_f.close()
steps_f.close()
prep_f.close()

print('CSV files closed.')
print(f'  {MEMBERS_CSV}')
print(f'  {STEPS_CSV}')
print(f'  {PREP_CSV}')

## Summary Statistics

In [ ]:
steps_df   = pd.read_csv(STEPS_CSV)
members_df = pd.read_csv(MEMBERS_CSV)

print('=== Step Summary ===')
print(steps_df[['step','dt_hours','valid_members','total_members',
                 'obs_area_ac','analysis_area_ac','elapsed_s']].to_string(index=False))

print(f'\nMean valid member rate : {members_df.farsite_ok.mean()*100:.1f}%')
print(f'Total FARSITE runs     : {len(members_df)}')
print(f'Total elapsed          : {steps_df.elapsed_s.sum():.1f}s')

## Visualise Results

In [ ]:
fig, axes = plt.subplots(1, len(results['step']), figsize=(6*len(results['step']), 5))
if len(results['step']) == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    t = results['step'][i]
    gpd.GeoSeries([results['initial_geom'][i]]).plot(ax=ax, color='blue',  alpha=0.3, label='Prior')
    gpd.GeoSeries([results['observed_geom'][i]]).plot(ax=ax, color='red',   alpha=0.3, label='Observed')
    gpd.GeoSeries([results['analysis_geom'][i]]).plot(ax=ax, color='green', alpha=0.4, label='Analysis')
    ax.set_title(f'Step {t}->{t+1}\n{results["t_end"][i].strftime("%Y-%m-%d %H:%M UTC")}')
    ax.legend(fontsize=8)
    ax.set_aspect('equal')

plt.tight_layout()
plt.savefig(LOG_DIR / f'{FIRE_NAME}_perimeters.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')